# Testbench: residual / ET-weighted contrast synthesis

Evaluates any number of trained runs on the **same batch**, in the **same intensity space**,
with the **same metrics** the training loop logs. Handles the three things that make these
runs awkward to compare naively:

1. **Residual mode.** `net(X)` returns `T1ce - T1`, not `T1ce`. The anchor channel is added
   back here, so every column below is a T1ce-domain image regardless of how it was trained.
2. **Per-run `scales`.** Each run divides its inputs by its own `scales`. The loader here is
   built **unscaled**, and each run's scaling is applied on the way in and undone on the way
   out -- so runs with different (or no) `scales` land in one comparable space.
3. **Tuple returns.** CDLNet-family models return `(x_hat, dS, z)`; U-Nets return a tensor.

Metrics are computed on the brain-masked, z-scored images exactly as `train_synthesis` does,
so the numbers here line up with what you see in wandb. `et_psnr` is PSNR inside the
enhancing tumor -- the number that actually moves when `lam_et` changes.

**Prereqs:** each run's saved `config.json` (it carries `paths.ckpt`) and its `net.ckpt`.
Runs whose config is missing are skipped. Edit `os.chdir` and the `RUNS` list.

In [ ]:
import os, json
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

os.chdir('/scratch/ee2178/ImMAP')          # <-- EDIT to your repo root

# ---- the runs to compare: (label, saved config.json) --------------------------------
# Mixing ET-weighted runs with their un-weighted baselines is the point: everything below
# reads each run's OWN config, so runs that predate `scales` / `et_mask` / `lam_et` work
# unchanged. Missing paths are dropped with a warning rather than raising.
RUNS = [
    # --- reference: the established e2e synthesis U-Net, direct T1ce, no ET weighting --
    ("UNet CCL-pretrain",      "trained_nets/brats/Synth_T1ce_Pretrain_VGG_CosLR/config.json"),
    # ("UNet no-pretrain",     "trained_nets/brats/Synth_T1ce_NoPretrain_VGG_CosLR/config.json"),
    # --- baseline: trained BEFORE ET weighting (lam_et absent, no `scales`) -----------
    ("DT-CDLNet dS (no ET)",   "trained_nets/brats/DT_CDLNet_T1ce/config.json"),
    # --- ET-weighted ------------------------------------------------------------------
    ("UNet resid (ET)",        "trained_nets/brats/Residual_Unet_T1ce_ET/config.json"),
    ("DT-CDLNet dS (ET)",      "trained_nets/brats/DT_CDLNet_dS_T1ce_ET/config.json"),
    ("DT-CDLNet noDS (ET)",    "trained_nets/brats/Residual_DT_CDLNet_noDS_T1ce_ET/config.json"),
    # add lam_et sweep arms here as you run them
    #
    # NOTE: the un-weighted residual U-Net baseline is gone -- the ET run was launched into
    # its directory. If you want that control back, train one arm with lam_et: 0 and add it
    # here; section 9 pairs it with the ET run automatically. Without it, the U-Net rows
    # below are absolute numbers only, with nothing to attribute the ET weighting against.
]
SPLIT        = "val"
CROP         = 192      # 192 = 6*32, divisible by the U-Net's 2**num_pool_layers -> no padding
BATCH        = 8
SEED         = 0

# ---- display / selection knobs ------------------------------------------------------
ONLY_TUMOR   = True     # keep ONLY slices that contain enhancing tumor. Roughly half of the
                        # BraTS slices in the kept z-band have none, and on those every ET
                        # metric is either 0 or undefined -- so they add nothing to a
                        # comparison about enhancement while diluting the averages.
                        # Applies to BOTH the panels and the aggregate table below.
SHOW_CONTOUR = False    # outline the ET mask on every panel. Informative for locating the
                        # tumor, visually noisy once you know where it is.
CONTOUR_LW   = 0.6      # linewidth when SHOW_CONTOUR is on

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from training.common import load_model, apply_loss_mask, region_psnr, region_loss
from training.metrics import compute_metrics
from training.losses import LOSS_REGISTRY
import datasets                                   # registers loaders
from datasets.registry import build_loader

missing = [(l, p) for l, p in RUNS if not os.path.exists(p)]
RUNS = [(l, p) for l, p in RUNS if os.path.exists(p)]
for l, p in missing:
    print(f"  [skip] {l}: no {p}")
assert RUNS, "none of the configs in RUNS exist -- check the paths / os.chdir"

# family = everything except lam_et. Runs sharing a family are the SAME architecture and
# target, differing only in ET weighting, so section 9 can difference them directly.
def family(c):
    return (c["model"]["type"], bool(c["training"].get("residual_mode", False)),
            bool(c["model"]["params"].get("use_dS", False)))

META = {}
print(f"device={device}  |  {len(RUNS)} run(s) found\n")
print(f"  {'label':24s} {'model':10s} {'resid':6s} {'lam_et':>7s} {'use_dS':7s} scales")
for lbl, p in RUNS:
    c = json.load(open(p))
    t, m = c["training"], c["model"]["params"]
    META[lbl] = dict(cfg=c, lam_et=float(t.get("lam_et", 0.0)), family=family(c),
                     residual=bool(t.get("residual_mode", False)))
    print(f"  {lbl:24s} {c['model']['type']:10s} "
          f"{str(t.get('residual_mode', False)):6s} {META[lbl]['lam_et']:7g} "
          f"{str(m.get('use_dS', 'n/a')):7s} {c['data']['train'].get('scales')}")

## 1. One shared, **unscaled** batch

The loader is built with `scales=None` so it yields raw z-scored data. Each run's own
`scales` are applied inside `predict_t1ce` below and undone on the output -- that is what
makes runs with different scaling comparable. `et_mask=True` gives the fourth item.

All runs must agree on `input_idx` / `target_idx` for a shared batch to be meaningful; that
is asserted rather than assumed.

`ONLY_TUMOR` filters at the SLICE level, not the batch level -- a batch is a mix, so dropping
whole batches would still leave tumour-free slices in the panels. `collect` keeps drawing
until enough qualifying slices accumulate.

In [ ]:
ref = json.load(open(RUNS[0][1]))
base = dict(ref["data"][SPLIT])
base.update(name="synthesis", scales=None, et_mask=True, center_crop=CROP,
            random_flips=False, num_workers=0, batch_size=BATCH)
base.pop("crop_size", None)                    # center crop only: deterministic

# a shared batch is only meaningful if every run reads the same channels
for lbl, p in RUNS[1:]:
    d = json.load(open(p))["data"][SPLIT]
    assert list(d["input_idx"]) == list(base["input_idx"]), f"{lbl}: input_idx differs"
    assert list(d["target_idx"]) == list(base["target_idx"]), f"{lbl}: target_idx differs"

def collect(n_slices, only_tumor, seed, max_batches=200):
    """Gather `n_slices` slices, optionally keeping only those that contain ET.

    Filtering has to happen at the SLICE level, not the batch level: a batch is a mix, so
    dropping whole batches would still leave tumour-free slices in the panels and in the
    averages. Draws successive batches until enough qualifying slices accumulate.
    """
    torch.manual_seed(seed)
    ld = build_loader(base, shuffle=True, drop_last=False)
    parts, got, seen = [], 0, 0
    for bi, batch in enumerate(ld):
        if bi >= max_batches or got >= n_slices:
            break
        bX, by, bm, bet = batch
        seen += bX.shape[0]
        keep = ((bet.flatten(1).sum(1) > 0) if only_tumor
                else torch.ones(bX.shape[0], dtype=torch.bool))
        if keep.any():
            parts.append(tuple(t[keep] for t in (bX, by, bm, bet)))
            got += int(keep.sum())
    if not parts:
        raise RuntimeError(f"no slices found (only_tumor={only_tumor}) in {max_batches} batches")
    out = tuple(torch.cat([p[i] for p in parts])[:n_slices].to(device) for i in range(4))
    if out[0].shape[0] < n_slices:
        print(f"[warn] asked for {n_slices} slices, found {out[0].shape[0]}")
    return out, seen

(X, y, mask, et), seen = collect(BATCH, ONLY_TUMOR, SEED)

n_et = int((et.flatten(1).sum(1) > 0).sum())
print(f"X {tuple(X.shape)}  y {tuple(y.shape)}  mask {tuple(mask.shape)}  et {tuple(et.shape)}")
print(f"ONLY_TUMOR={ONLY_TUMOR}: kept {X.shape[0]} slice(s) out of {seen} scanned; "
      f"{n_et}/{X.shape[0]} contain ET")
print(f"brain fraction {float(mask.mean()):.3f}   ET fraction {float(et.mean()):.5f}")
if not ONLY_TUMOR and n_et < X.shape[0]:
    print(f"[note] {X.shape[0]-n_et} slice(s) have no tumor; their et_psnr is 0 by convention "
          f"and will drag the averages down. Set ONLY_TUMOR=True to exclude them.")

## 2. Predict, in a common space

`predict_t1ce` mirrors the training forward pass exactly, then inverts it:

```
X_scaled  = X_raw / scales[input_idx]        # what the net was trained on
pred      = net(X_scaled)                    # residual, or T1ce, per the run's config
t1ce_s    = pred + X_scaled[:, src_idx]      # residual mode only
t1ce_raw  = t1ce_s * scales[target_idx]      # back to the shared, unscaled space
```

The residual is reconstructed **in the run's own scaled space** before undoing the scale.
Doing it the other way round is only equivalent when every channel shares one scale factor,
which is not something the config guarantees.

In [ ]:
def load_run(cfg_path):
    cfg = json.load(open(cfg_path))
    net = load_model(cfg_path, device=device)         # build_model + ckpt + eval()
    return net, cfg

@torch.no_grad()
def predict_t1ce(net, cfg, X_raw):
    """-> (t1ce_raw, residual_raw_or_None, dS_or_None), all in the shared unscaled space."""
    tr, tt = cfg["data"]["train"], cfg["training"]
    inp, tgt = list(tr["input_idx"]), int(tr["target_idx"][0])
    scales = tr.get("scales")

    if scales is not None:
        s_in = torch.tensor([float(scales[i]) for i in inp], device=X_raw.device).view(1, -1, 1, 1)
        s_y = float(scales[tgt])
    else:
        s_in, s_y = torch.ones(1, len(inp), 1, 1, device=X_raw.device), 1.0
    Xs = X_raw / s_in

    out = net(Xs)
    dS = None
    if isinstance(out, (tuple, list)):                # DTCDLNet (x_hat, dS, z) / GroupCDL (x_hat, z)
        pred, rest = out[0], out[1:]
        if len(rest) >= 2:                            # (dS, z): dS is the enhancement map
            dS = rest[0] * s_y
    else:
        pred = out

    if tt.get("residual_mode"):
        i = int(tt.get("residual_src_idx", 0))
        src = Xs[:, i:i + 1]
        return (pred + src) * s_y, pred * s_y, dS
    return pred * s_y, None, dS

preds = {}
for lbl, p in RUNS:
    net, cfg = load_run(p)
    t1ce, resid, dS = predict_t1ce(net, cfg, X)
    preds[lbl] = dict(t1ce=t1ce, resid=resid, dS=dS, cfg=cfg, net=net)
    extra = []
    if resid is not None: extra.append("residual->T1ce")
    if dS is not None:    extra.append(f"dS max={float(dS.abs().max()):.3f}")
    print(f"{lbl:24s} ok   {' | '.join(extra) if extra else 'direct'}")

## 3. Metrics

Same masking and same functions as `train_synthesis`, so these are directly comparable to
the wandb curves. `et_psnr` takes the MSE over ET pixels only; it is 0 when a batch has no
enhancing tumor.

In [ ]:
def eval_one(y_true, y_pred, mask, et, psnr_only=False):
    yt, yp = apply_loss_mask(y_true, y_pred, mask, True)
    m = {k: float(v) for k, v in compute_metrics(yt, yp, psnr_only=psnr_only).items()}
    m["et_psnr"] = float(region_psnr(yt, yp, et))
    m["l1"] = float(LOSS_REGISTRY["magnitude-l1"](yp, yt, None))
    m["et_l1"] = float(region_loss(LOSS_REGISTRY["magnitude-l1"], yp, yt, et))
    return m

# T1 anchor: the trivial "predict no enhancement at all" answer, and the bar every model has
# to clear on et_psnr. Derived from the STORED channel order (flair=0, t1=1, t1ce=2, t2=3)
# rather than from any one run's residual_src_idx -- non-residual runs do not have that key,
# and reading it off RUNS[0] would silently change the baseline when the list is reordered.
STORED_T1 = 1
inp = list(base["input_idx"])
src_idx = inp.index(STORED_T1) if STORED_T1 in inp else 0
t1_anchor = X[:, src_idx:src_idx + 1]
print(f"anchor = input position {src_idx} (stored channel {inp[src_idx]} = t1)\n")

rows = {"T1 anchor (baseline)": eval_one(y, t1_anchor, mask, et)}
for lbl in preds:
    rows[lbl] = eval_one(y, preds[lbl]["t1ce"], mask, et)

hdr = ["psnr", "ssim", "nrmse", "et_psnr", "l1", "et_l1"]
print(f"{'method':26s} {'lam_et':>7s} " + " ".join(f"{h:>9s}" for h in hdr))
print("-" * (35 + 10 * len(hdr)))
for lbl, m in rows.items():
    lam = f"{META[lbl]['lam_et']:7g}" if lbl in META else " " * 7
    print(f"{lbl:26s} {lam} " + " ".join(f"{m.get(h, float('nan')):9.4f}" for h in hdr))

## 4. Side by side, T1ce domain

Shared display window from the T1 anchor + ground truth so the columns are comparable.
Per-panel labels carry global PSNR and ET PSNR; the green outline is the ET mask.

In [ ]:
def win(*imgs, m):
    v = torch.cat([im[m.bool().expand_as(im)].flatten() for im in imgs]).float().cpu()
    return float(v.quantile(0.01)), float(v.quantile(0.99))

def outline(ax_, et_slice, color="lime", lw=None):
    """Draw the ET contour, unless SHOW_CONTOUR is off. One place, so the toggle cannot
    end up applying to some panels and not others."""
    if not SHOW_CONTOUR:
        return
    a = et_slice.detach().cpu().numpy()
    if a.max() > 0:
        ax_.contour(a, levels=[0.5], colors=color, linewidths=lw or CONTOUR_LW)

cols = [("T1 anchor", t1_anchor)] + [(l, preds[l]["t1ce"]) for l in preds] + [("T1ce GT", y)]
nshow = min(4, y.shape[0])
vmn, vmx = win(t1_anchor[:nshow], y[:nshow], m=mask[:nshow])

fig, ax = plt.subplots(nshow, len(cols), figsize=(2.3 * len(cols), 2.6 * nshow), squeeze=False)
for i in range(nshow):
    for j, (name, img) in enumerate(cols):
        ax[i, j].imshow((img[i, 0] * mask[i, 0]).cpu().numpy(), cmap="gray", vmin=vmn, vmax=vmx)
        if i == 0:
            ax[i, j].set_title(name, fontsize=9)
        if name != "T1ce GT":
            m1 = eval_one(y[i:i+1], img[i:i+1], mask[i:i+1], et[i:i+1], psnr_only=True)
            ax[i, j].set_xlabel(f"{m1['psnr']:.1f} dB | ET {m1['et_psnr']:.1f}", fontsize=7)
        outline(ax[i, j], et[i, 0])
        ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
plt.tight_layout(); plt.show()

## 5. Residual maps (`T1ce - T1`), diverging scale

The quantity the residual runs are actually trained on. `bwr` with a symmetric window --
white is zero, red positive, blue negative -- on **one shared scale** across all columns, so
over- and under-shoot are readable. A gray colormap would render a mostly-zero signed map as
flat mid-gray and hide the sign entirely. This matches the `val/delta` panel in wandb.

Direct-prediction runs get a residual too, by subtracting the anchor from their T1ce output,
so the column is meaningful for every run.

In [ ]:
gt_resid = y - t1_anchor
res_cols = [("GT residual", gt_resid)] + [(l, preds[l]["t1ce"] - t1_anchor) for l in preds]

r = torch.cat([(c * mask).abs().flatten() for _, c in res_cols]).float()
v = float(torch.quantile(r[r > 0], 0.995)) if float(r.max()) > 0 else 1.0

fig, ax = plt.subplots(nshow, len(res_cols), figsize=(2.3 * len(res_cols), 2.6 * nshow),
                       squeeze=False)
for i in range(nshow):
    for j, (name, img) in enumerate(res_cols):
        im = ax[i, j].imshow((img[i, 0] * mask[i, 0]).cpu().numpy(), cmap="bwr", vmin=-v, vmax=v)
        if i == 0:
            ax[i, j].set_title(name, fontsize=9)
        outline(ax[i, j], et[i, 0], color="k", lw=0.5)
        ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
fig.colorbar(im, ax=ax, shrink=0.6, label=f"residual (shared scale +/-{v:.2f})")
plt.suptitle("T1ce - T1, white = 0", y=1.01, fontsize=10); plt.show()

## 6. Enhancing-tumor closeup

Zooms the slice with the most ET. This is where `lam_et` is supposed to pay off, and where
the global PSNR column is least informative.

In [ ]:
areas = et.flatten(1).sum(1)
if float(areas.max()) == 0:
    print("no ET in this batch -- skipping closeup (try another SEED)")
else:
    i = int(areas.argmax())
    ys, xs = torch.nonzero(et[i, 0], as_tuple=True)
    pad = 24
    y0, y1 = max(0, int(ys.min()) - pad), min(et.shape[-2], int(ys.max()) + pad)
    x0, x1 = max(0, int(xs.min()) - pad), min(et.shape[-1], int(xs.max()) + pad)
    sl = (slice(y0, y1), slice(x0, x1))

    zcols = [("T1 anchor", t1_anchor)] + [(l, preds[l]["t1ce"]) for l in preds] + [("T1ce GT", y)]
    fig, ax = plt.subplots(1, len(zcols), figsize=(2.6 * len(zcols), 3.0), squeeze=False)
    for j, (name, img) in enumerate(zcols):
        ax[0, j].imshow(img[i, 0][sl].cpu().numpy(), cmap="gray", vmin=vmn, vmax=vmx)
        outline(ax[0, j], et[i, 0][sl], lw=0.8)
        ttl = name
        if name != "T1ce GT":
            mm = eval_one(y[i:i+1], img[i:i+1], mask[i:i+1], et[i:i+1], psnr_only=True)
            ttl += f"\nET {mm['et_psnr']:.1f} dB"
        ax[0, j].set_title(ttl, fontsize=9); ax[0, j].set_xticks([]); ax[0, j].set_yticks([])
    plt.suptitle(f"slice {i}: ET = {int(areas[i])} px "
                 f"({float(et[i].mean()):.4%} of the slice)", fontsize=10)
    plt.tight_layout(); plt.show()

## 7. (optional) dS enhancement map

Only for DT-CDLNet runs built with `use_dS=True`; runs with the branch off return zeros by
construction and are skipped. Watch for the two failure modes the loss cannot see: `dS`
collapsing to zero, or `dS` absorbing the whole image so the decomposition is meaningless.
The useful number is how much of `dS` lands inside the ET contour.

In [ ]:
ds_runs = [(l, d) for l, d in preds.items()
           if d["dS"] is not None and float(d["dS"].abs().max()) > 0]
if not ds_runs:
    print("no run in RUNS has an active dS branch (use_dS=False, or a non-CDLNet model)")
else:
    fig, ax = plt.subplots(len(ds_runs), nshow, figsize=(2.6 * nshow, 2.7 * len(ds_runs)),
                           squeeze=False)
    for r_, (lbl, d) in enumerate(ds_runs):
        dS = d["dS"] * mask
        vv = float(torch.quantile(dS.abs().flatten().float(), 0.999))
        inside = (float((dS[et.bool().expand_as(dS)] > 0).float().mean())
                  if float(et.sum()) else float("nan"))
        for i in range(nshow):
            ax[r_, i].imshow(dS[i, 0].cpu().numpy(), cmap="magma", vmin=0, vmax=max(vv, 1e-8))
            outline(ax[r_, i], et[i, 0], color="cyan")
            ax[r_, i].set_xticks([]); ax[r_, i].set_yticks([])
        # is dS landing ON the tumor, or scattered over the whole brain?
        ax[r_, 0].set_title(f"{lbl}: dS active inside ET = {inside:.2f}, "
                            f"overall = {float((dS > 0).float().mean()):.3f}",
                            fontsize=8, loc="left")
    plt.tight_layout(); plt.show()

## 8. Aggregate over several batches

Single-batch numbers are noisy, `et_psnr` especially -- ET is well under 1% of a slice and
absent from many of them, so the per-batch ET pixel count swings several-fold. Average over
enough batches before drawing conclusions. Batches with no enhancing tumor are excluded from
the `et_psnr` mean rather than counted as 0, which would drag it down for reasons unrelated
to model quality. `n with ET` reports how many actually contributed.

In [ ]:
N_EVAL_SLICES = 96          # pooled, then chunked into BATCH-sized forward passes

(eX, ey, em, eet), e_seen = collect(N_EVAL_SLICES, ONLY_TUMOR, SEED + 1)
print(f"evaluating on {eX.shape[0]} slice(s) (ONLY_TUMOR={ONLY_TUMOR}, {e_seen} scanned)")

agg = {lbl: [] for lbl in ["T1 anchor (baseline)"] + list(preds)}
n_chunks = n_with_et = 0
for i in range(0, eX.shape[0], BATCH):
    sl = slice(i, i + BATCH)
    bX, by, bm, bet = eX[sl], ey[sl], em[sl], eet[sl]
    has_et = float(bet.sum()) > 0
    n_chunks += 1; n_with_et += int(has_et)
    outs = {"T1 anchor (baseline)": bX[:, src_idx:src_idx + 1]}
    for lbl, d in preds.items():
        outs[lbl] = predict_t1ce(d["net"], d["cfg"], bX)[0]
    for lbl, o in outs.items():
        m = eval_one(by, o, bm, bet)
        # NaN, not 0, when a chunk has no tumor: region_psnr returns 0 for an empty region,
        # and averaging that in would report "0 dB of tumor fidelity" for slices that simply
        # have no tumor. nanmean below then skips them. (The training loop does NOT do this,
        # so its val/et_psnr reads lower than the number here.)
        agg[lbl].append((m["psnr"], m["ssim"], m["et_psnr"] if has_et else np.nan, m["l1"]))

print(f"{'method':26s} {'PSNR':>9} {'SSIM':>9} {'ET PSNR':>9} {'L1':>9}   "
      f"({n_chunks} chunks, {n_with_et} with ET)")
print("-" * 78)
summary = {}
for lbl, v in agg.items():
    a = np.array(v, dtype=float)
    summary[lbl] = (float(np.mean(a[:, 0])), float(np.mean(a[:, 1])),
                    float(np.nanmean(a[:, 2])), float(np.mean(a[:, 3])))
    print(f"{lbl:26s} " + " ".join(f"{x:9.4f}" for x in summary[lbl]))
if ONLY_TUMOR:
    print("\nONLY_TUMOR=True: PSNR/SSIM/L1 above are over TUMOUR-BEARING slices only, so they")
    print("are not comparable with a whole-split number. Set ONLY_TUMOR=False for that.")

## 9. ET weighting: control vs treatment

Pairs each ET-weighted run against the un-weighted run of the **same architecture and
target** (same model type, same `residual_mode`, same `use_dS`) and reports the deltas. This
is the comparison that says whether `lam_et` earned anything, and it only works because both
arms are evaluated here on the same batches, in the same space, with the same metrics.

Read `d(ET PSNR)` against `d(PSNR)`: ET weighting is meant to buy the first at the cost of
the second. A run that gained neither did nothing; one that lost both is over-weighted.

In [ ]:
fams = {}
for lbl in preds:
    fams.setdefault(META[lbl]["family"], []).append(lbl)

paired = 0
for fam, labels in fams.items():
    ctrl = [l for l in labels if META[l]["lam_et"] == 0]
    treat = [l for l in labels if META[l]["lam_et"] > 0]
    if not ctrl or not treat:
        continue
    paired += 1
    c = ctrl[0]
    mtype, resid, useds = fam
    print(f"\n{mtype}  residual={resid}  use_dS={useds}")
    print(f"  {'arm':26s} {'lam_et':>7s} {'PSNR':>8} {'d':>7} {'ET PSNR':>9} {'d':>7} {'SSIM':>8}")
    cp, cs, ce, _ = summary[c]
    print(f"  {c + ' (control)':26s} {META[c]['lam_et']:7g} {cp:8.3f} {'--':>7} "
          f"{ce:9.3f} {'--':>7} {cs:8.4f}")
    for t in sorted(treat, key=lambda l: META[l]["lam_et"]):
        tp, ts, te, _ = summary[t]
        print(f"  {t:26s} {META[t]['lam_et']:7g} {tp:8.3f} {tp - cp:+7.3f} "
              f"{te:9.3f} {te - ce:+7.3f} {ts:8.4f}")
if paired == 0:
    print("no control/treatment pair in RUNS -- add a lam_et=0 run of the same architecture")
    print("(same model type, residual_mode and use_dS) to enable this comparison")

## 10. The `lam_et` trade-off

Global PSNR against ET PSNR. There is no single best point -- you are choosing how much
whole-brain fidelity to give up for tumor fidelity, so read the upper-right frontier and
pick from it. Add `lam_et` sweep arms to `RUNS` and this becomes the plot you tune on.

In [ ]:
# A lam_et=0 run is only a CONTROL if its own architecture also appears with lam_et>0.
# A standalone un-weighted run (the CCL-pretrain U-Net) is a REFERENCE point instead --
# same axes, but nothing in the sweep is being attributed to it, so it gets its own marker.
controls = set()
for _f, _labels in fams.items():
    if (any(META[l]["lam_et"] > 0 for l in _labels)
            and any(META[l]["lam_et"] == 0 for l in _labels)):
        controls.update(_labels)

fig, ax = plt.subplots(figsize=(7.4, 5.2))
for lbl, (p_, s_, e_, l_) in summary.items():
    is_base = lbl == "T1 anchor (baseline)"
    lam = META.get(lbl, {}).get("lam_et", 0.0)
    kw = dict(s=130, zorder=3)
    if is_base:
        kw.update(marker="X", c="0.5")                                   # copy T1 verbatim
    elif lam > 0:
        kw.update(marker="o", c="tab:red")                               # ET-weighted
    elif lbl in controls:
        kw.update(marker="o", facecolors="none", edgecolors="tab:blue",  # paired control
                  linewidths=1.8)
    else:
        kw.update(marker="s", facecolors="none", edgecolors="tab:green", # standalone reference
                  linewidths=1.8)
    ax.scatter(p_, e_, **kw)
    ax.annotate(lbl, (p_, e_), fontsize=8, xytext=(6, 4), textcoords="offset points")
# arrow from each control to its ET-weighted counterpart: the direction IS the trade-off
for fam, labels in fams.items():
    ctrl = [l for l in labels if META[l]["lam_et"] == 0 and l in summary]
    for t in [l for l in labels if META[l]["lam_et"] > 0 and l in summary]:
        if ctrl:
            cp, _, ce, _ = summary[ctrl[0]]
            tp, _, te, _ = summary[t]
            ax.annotate("", xy=(tp, te), xytext=(cp, ce),
                        arrowprops=dict(arrowstyle="->", color="0.6", lw=1.2))
if "T1 anchor (baseline)" in summary:
    ax.axhline(summary["T1 anchor (baseline)"][2], color="0.7", ls="--", lw=1, zorder=1)
ax.set_xlabel("global PSNR (dB) -- whole brain")
ax.set_ylabel("ET PSNR (dB) -- enhancing tumor")
ax.set_title("Upper-right is better; the frontier is the choice", fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Reading this plot:")
print("  X grey       copy T1 verbatim -- the do-nothing answer")
print("  green square standalone reference (e.g. the CCL-pretrain U-Net): not part of the")
print("               ET experiment, just the bar the new approaches have to clear")
print("  blue hollow  lam_et=0 control, paired with an ET-weighted run of the same architecture")
print("  red filled   ET-weighted")
print("  arrows       control -> its ET-weighted counterpart; the direction is the trade-off.")
print("  * anything at or below the dashed line has not beaten 'copy T1', i.e. has learned")
print("    nothing about enhancement, whatever its global PSNR says")
print("  * an arrow pointing UP and slightly LEFT is lam_et working as intended;")
print("    straight LEFT (or down) means it is only costing you fidelity")
print("  * a large ET gain for <0.5 dB of global PSNR is usually a good trade")